# Newspaper Annotation Review

Runs the **Generator / Critic** annotation pipeline on a newspaper page and
displays the per-round visualisations side-by-side so you can see how the
critic refines the initial bounding-box pass.

**Models used**
- Generator: `gemini-2.5-flash` (free-tier key `GEMINI_API_KEY`)
- Critic: `gemini-2.5-pro` (paid key `GEMINI_PRO_API_KEY`)

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Make the src/ package importable when running from the notebooks/ directory
repo_root = Path().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from newspapers.segmentation.annotate import annotate_page, annotate_page_structured
from newspapers.segmentation.structure import analyse_page_structure, draw_column_bounds

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

# Image to annotate (change to any .jpg in data/processed/)
IMAGE_PATH = repo_root / "data" / "processed" / "bib13991099_19000124_0_10721a_0002.jpg"

GENERATOR_MODEL  = "gemini-2.5-flash"
CRITIC_MODEL     = "gemini-2.5-pro"
CRITIQUE_ROUNDS  = 1          # increase for more refinement passes

LABELS_DIR = repo_root / "data" / "annotations" / "labels" / "train"
IMAGES_DIR = repo_root / "data" / "annotations" / "images" / "train"
VIS_DIR    = repo_root / "data" / "annotations" / "visualizations"

OVERWRITE = True  # set False to skip images already annotated

In [ ]:
# ── Run annotation ────────────────────────────────────────────────────────────
regions = annotate_page(
    IMAGE_PATH,
    labels_dir=LABELS_DIR,
    images_dir=IMAGES_DIR,
    vis_dir=VIS_DIR,
    generator_model=GENERATOR_MODEL,
    critic_model=CRITIC_MODEL,
    critique_rounds=CRITIQUE_ROUNDS,
    overwrite=OVERWRITE,
)
print(f"Final region count: {len(regions)}")

In [ ]:
# ── Round-by-round visualisation ─────────────────────────────────────────────
stem = IMAGE_PATH.stem

round_images = []
for r in range(CRITIQUE_ROUNDS + 1):          # r0 … rN
    p = VIS_DIR / f"{stem}_r{r}_vis.png"
    if p.exists():
        round_images.append((f"Round {r}", p))

final_vis = VIS_DIR / f"{stem}_vis.png"
if final_vis.exists():
    round_images.append(("FINAL", final_vis))

n_cols = len(round_images)
if n_cols == 0:
    print("No visualisation files found — run the annotation cell first.")
else:
    fig, axes = plt.subplots(1, n_cols, figsize=(8 * n_cols, 14))
    if n_cols == 1:
        axes = [axes]
    for ax, (title, img_path) in zip(axes, round_images):
        ax.imshow(Image.open(img_path))
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Stats: region counts & class distribution per round ──────────────────────
stats_path = VIS_DIR / f"{stem}_stats.json"

if not stats_path.exists():
    print("No stats file found — run the annotation cell first.")
else:
    stats = json.loads(stats_path.read_text())
    print(f"Image : {stats['image']}")
    print(f"Annotated at : {stats['annotated_at']}")
    print(f"Generator : {stats['generator_model']}")
    print(f"Critic    : {stats['critic_model']}")
    print()

    # Build a tidy DataFrame with one row per (round × class)
    rows = []
    for rnd in stats["rounds"]:
        for cls, cnt in rnd["class_counts"].items():
            rows.append(
                {
                    "round": f"R{rnd['round']} ({rnd['role']})",
                    "class": cls,
                    "count": cnt,
                }
            )

    df = pd.DataFrame(rows)
    pivot = df.pivot_table(index="class", columns="round", values="count", fill_value=0)

    print("Region counts per class per round:")
    display(pivot)  # noqa: F821  (Jupyter built-in)

    # Bar chart
    ax = pivot.plot(
        kind="bar",
        figsize=(10, 4),
        title="Class distribution — Generator vs Critic rounds",
        rot=25,
    )
    ax.set_ylabel("# regions")
    ax.legend(title="Round")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Batch annotation (all .jpg files in data/processed/) ─────────────────────
# Uncomment and run this cell to process every image in the processed folder.

# from tqdm.notebook import tqdm
# from newspapers.segmentation.annotate import annotate_directory
#
# summary = annotate_directory(
#     repo_root / "data" / "processed",
#     labels_dir=LABELS_DIR,
#     images_dir=IMAGES_DIR,
#     vis_dir=VIS_DIR,
#     generator_model=GENERATOR_MODEL,
#     critic_model=CRITIC_MODEL,
#     critique_rounds=CRITIQUE_ROUNDS,
#     overwrite=OVERWRITE,
# )
# for stem, count in summary.items():
#     status = f"{count} regions" if count >= 0 else "FAILED"
#     print(f"  {stem}: {status}")

---
## Column-Aware (Structured) Annotation

Uses `analyse_page_structure` to detect column boundaries with classical CV
(projection profiles + vertical-rule detection), then sends each column strip
to Gemini separately. Results are merged back to full-page 0–1000 coordinates
and deduplicated by IoU.

Key parameters to tune:
- `N_COLUMNS` – expected column count (default 8 for this paper)
- `OVERLAP_FRAC` – fractional strip overlap on each side (0.05 = 5 %)


In [ ]:
# ── Structured config ──────────────────────────────────────────────────────
N_COLUMNS    = 8
OVERLAP_FRAC = 0.05   # tune: wider overlap catches more cross-column elements

# Use the high-res PNG for structure analysis if available
PNG_PATH  = IMAGE_PATH.with_suffix('.png')
STRUCT_SRC = PNG_PATH if PNG_PATH.exists() else IMAGE_PATH

# ── Detect page structure ──────────────────────────────────────────────────
column_bounds, strips, profile, skew_angle = analyse_page_structure(
    STRUCT_SRC, n_columns_hint=N_COLUMNS
)

print(f'Skew: {skew_angle:.2f} deg')
print(f'Column boundaries (n={len(column_bounds)}): {column_bounds}')
print(f'Strips created: {len(strips)}')

# ── Draw column boundary overlay ───────────────────────────────────────────
display_img = Image.open(STRUCT_SRC)
vis_img     = draw_column_bounds(display_img, column_bounds, profile=profile)

display_w = 900
scale     = display_w / vis_img.width
vis_small = vis_img.resize((display_w, int(vis_img.height * scale)), Image.LANCZOS)

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(vis_small)
ax.axis('off')
ax.set_title(f'Column boundaries detected (n={len(column_bounds)+1} columns)')
plt.tight_layout()
plt.show()


In [ ]:
# ── Strip browser ─────────────────────────────────────────────────────────
col_strips = [s for s in strips if s.strip_id not in ('masthead', 'full')]

fig, axes = plt.subplots(1, len(col_strips), figsize=(3 * len(col_strips), 8))
if len(col_strips) == 1:
    axes = [axes]
for ax, strip in zip(axes, col_strips):
    img   = Image.open(strip.image_path)
    thumb = img.resize((img.width, min(img.height, 800)), Image.LANCZOS)
    ax.imshow(thumb)
    ax.set_title(f'col {strip.column_index}\n{strip.strip_width}px', fontsize=8)
    ax.axis('off')
plt.suptitle('Column strips (with overlap, pre-annotation)', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Run structured annotation ──────────────────────────────────────────────
# Calls Gemini separately for each column strip; may take a few minutes.
regions_structured = annotate_page_structured(
    IMAGE_PATH,
    labels_dir=LABELS_DIR,
    images_dir=IMAGES_DIR,
    vis_dir=VIS_DIR,
    generator_model=GENERATOR_MODEL,
    critic_model=CRITIC_MODEL,
    critique_rounds=CRITIQUE_ROUNDS,
    n_columns_hint=N_COLUMNS,
    overlap_frac=OVERLAP_FRAC,
    overwrite=OVERWRITE,
)
print(f'Structured annotation — final region count: {len(regions_structured)}')

# ── Show the structured visualisation ─────────────────────────────────────
vis_path = VIS_DIR / f'{IMAGE_PATH.stem}_structured_vis.png'
if vis_path.exists():
    vis   = Image.open(vis_path)
    scale = 900 / vis.width
    vis_s = vis.resize((900, int(vis.height * scale)), Image.LANCZOS)
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(vis_s)
    ax.axis('off')
    ax.set_title('Structured annotation — merged output (red lines = column bounds)')
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Merge diagnostic — region provenance per strip ────────────────────────
stats_path = VIS_DIR / f'{IMAGE_PATH.stem}_structured_stats.json'

if not stats_path.exists():
    print('No structured stats file — run the annotation cell above first.')
else:
    stats = json.loads(stats_path.read_text())
    print(f"Image         : {stats['image']}")
    print(f"Annotated at  : {stats['annotated_at']}")
    print(f"Columns       : {stats['n_columns']}")
    print(f"Overlap frac  : {stats['overlap_frac']}")
    print(f"Final regions : {stats['final_region_count']}")
    print()

    strip_stats = stats.get('strip_stats', [])
    if strip_stats:
        df = pd.DataFrame(strip_stats)
        print(df.to_string(index=False))
        print()

        if 'strip_id' in df.columns and 'region_count' in df.columns:
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.bar(df['strip_id'].astype(str), df['region_count'])
            ax.set_xlabel('Strip')
            ax.set_ylabel('Region count')
            ax.set_title('Regions detected per strip (before merge)')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
